In [1]:
import sys
sys.path.append('../../../code/')
# stage1_train_classifier.py
import os
import pickle
import numpy as np
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from shared_util.seed import seed_everything
from shared_util.logger import Logger
from shared_util.file_handler import FileHandler

# 你现有的模块
from HolisticRCA.data_loader.mask_learning_data_loader import MaskLearningDataLoader
from HolisticRCA.util.data_handler import copy_batch_data  # 如果你需要

In [2]:
import math
import torch
import torch.nn as nn
from shared_util.evaluation_metrics import *
from sklearn.metrics import precision_recall_fscore_support
from HolisticRCA.data_loader.mask_learning_data_loader import MaskLearningDataLoader
from HolisticRCA.util.data_handler import copy_batch_data
from collections import defaultdict
import numpy as np


class PositionalEmbedding(nn.Module):
    def __init__(self, in_features, num_of_o11y_features, register_name):
        super(PositionalEmbedding, self).__init__()

        temp_in_features = in_features
        if temp_in_features % 2 == 1:
            temp_in_features += 1

        temp_num_of_o11y_features = num_of_o11y_features
        if temp_num_of_o11y_features % 2 == 1:
            temp_num_of_o11y_features += 1

        pe = torch.zeros(temp_in_features, temp_num_of_o11y_features).float()
        pe.require_grad = False

        position = torch.arange(0, temp_in_features).float().unsqueeze(1)
        div_term = (torch.arange(0, temp_num_of_o11y_features, 2).float() * -(math.log(10000.0) / temp_num_of_o11y_features)).exp()

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.transpose(1, 0)[:num_of_o11y_features, :in_features]
        self.register_buffer('pe', pe)

    def forward(self, x):
        return self.pe + x

In [3]:
class RepresentationLearning(nn.Module):
    def __init__(self, param_dict, meta_data):
        super().__init__()
        self.device_marker = nn.Parameter(torch.empty(0))
        self.meta_data = meta_data
        self.different_modal_mapping_dict = nn.ModuleDict()
        self.positional_embedding_dict = nn.ModuleDict()
        self.modal_transformer_encoder_layer_dict = nn.ModuleDict()
        for modal_type in self.meta_data['modal_types']:
            self.positional_embedding_dict[modal_type] = PositionalEmbedding(in_features=param_dict['window_size'],
                                                                             num_of_o11y_features=self.meta_data['o11y_length'][modal_type],
                                                                             register_name=f'{modal_type}_pe')
            self.different_modal_mapping_dict[modal_type] = nn.Linear(in_features=param_dict['window_size'],
                                                                      out_features=param_dict['orl_te_in_channels'])
            transformer_encoder_layer = nn.TransformerEncoderLayer(d_model=param_dict['orl_te_in_channels'],
                                                                   nhead=param_dict['orl_te_heads'],
                                                                   batch_first=True)
            self.modal_transformer_encoder_layer_dict[modal_type] = nn.TransformerEncoder(transformer_encoder_layer, num_layers=param_dict['orl_te_layers'])

    def forward(self, batch_data):
        for modal_type in self.meta_data['modal_types']:
            batch_data[f'x_{modal_type}'] = batch_data[f'x_{modal_type}'].to(self.device_marker.device) # (B, F, T)
            batch_data[f'x_{modal_type}'] = self.positional_embedding_dict[modal_type](batch_data[f'x_{modal_type}']).contiguous() # (B, F, T)
            batch_data[f'x_{modal_type}'] = self.different_modal_mapping_dict[modal_type](batch_data[f'x_{modal_type}']) # (B, F, C)
            # 注意,这里transformer不是编码时间,这里为了同时保持可观测特征的独立性并学习其相关性或因果关系,将每个可观测特征视为一个图节点进行全连接图表示学习
            batch_data[f'x_{modal_type}'] = self.modal_transformer_encoder_layer_dict[modal_type](batch_data[f'x_{modal_type}']) # (B, F, C)
        return batch_data

In [4]:
import torch
import torch.nn as nn


class FeatureIntegration(nn.Module):
    def __init__(self, param_dict, meta_data):
        super().__init__()
        self.device_marker = nn.Parameter(torch.empty(0))
        self.meta_data = meta_data

        self.ent_transformer_encoder_layer_dict = nn.ModuleDict()
        self.align_embedding_dict = nn.ModuleDict()
        self.ent_feature_align_dict = nn.ModuleDict()

        in_dim = param_dict['efi_in_dim']

        for ent_type in self.meta_data['ent_types']:
            all_ent_feature_length = 0
            for modal_type in self.meta_data['modal_types']:
                all_ent_feature_length += self.meta_data['max_ent_feature_num'][ent_type][modal_type]

            transformer_encoder_layer = nn.TransformerEncoderLayer(d_model=in_dim, nhead=param_dict['efi_te_heads'], batch_first=True)
            self.ent_transformer_encoder_layer_dict[ent_type] = nn.TransformerEncoder(transformer_encoder_layer, num_layers=param_dict['efi_te_layers'])

            self.ent_feature_align_dict[ent_type] = nn.Linear(all_ent_feature_length * in_dim, param_dict['efi_out_dim'])

    def forward(self, batch_data):
        batch_size = batch_data['y'].shape[0]

        x_ent = []
        for ent_type in self.meta_data['ent_types']:
            '''
                node: 0~5
                service: 6~15
                pod: 16~55
            '''
            for ent_index in range(self.meta_data['ent_type_index'][ent_type][0], self.meta_data['ent_type_index'][ent_type][1]):
                x = []
                for modal_type in self.meta_data['modal_types']:
                    feature_index_pair = self.meta_data['ent_features'][modal_type][ent_index][1]
                    # 从(B, F_modal, C)中,把相应实体的特征切分出来,变成(B, X, C)
                    modal_data = batch_data[f'x_{modal_type}'][:, feature_index_pair[0]:feature_index_pair[1], :]
                    # (B, X_ent_mod, C),同一个资源实体类型同一个模态的特征补齐成相同形状
                    padding = torch.zeros(batch_size, self.meta_data['max_ent_feature_num'][ent_type][modal_type] - modal_data.shape[1], modal_data.shape[2]).to(self.device_marker.device)
                    modal_data = torch.cat((modal_data, padding), 1)
                    x.append(modal_data)
                # x: (B, total_feature_num, C)
                # total_feature_num = max_ent_feature_num[node][metric] + max_ent_feature_num[node][trace] + max_ent_feature_num[node][log]
                x = torch.cat(x, dim=1)
                # 建模每个资源实体内部特征之间的关系 (B, total_feature_num, C)
                x = self.ent_transformer_encoder_layer_dict[ent_type](x)
                # (B, total_feature_num * C)
                x = x.view(batch_size, x.shape[1] * x.shape[2]).contiguous()
                # (B, hidden),把取出的每个资源实体的多模态特征,组织成一个整体的实体资源级的特征
                x = self.ent_feature_align_dict[ent_type](x)
                x_ent.append(x)
                
        x_ent = torch.stack(x_ent, dim=1)
        # (B, E, hidden)
        batch_data['x_ent'] = x_ent
        return batch_data

In [5]:
import torch
import torch.nn as nn
from HolisticRCA.model.common.GAT_net import GATNet
from HolisticRCA.util.ent_batch_graph import EntBatchGraph


class FeatureFusion(nn.Module):
    def __init__(self, param_dict, meta_data):
        super().__init__()
        self.device_marker = nn.Parameter(torch.empty(0))
        self.meta_data = meta_data
        self.GAT_net = GATNet(in_channels=param_dict['eff_in_dim'],
                              out_channels=param_dict['eff_GAT_out_channels'],
                              heads=param_dict['eff_GAT_heads'],
                              dropout=param_dict['eff_GAT_dropout'])
        self.linear_dict = nn.ModuleDict()
        for ent_type in self.meta_data['ent_types']:
            index_pair = self.meta_data['ent_fault_type_index'][ent_type]
            self.linear_dict[ent_type] = nn.Linear(param_dict['eff_GAT_out_channels'], index_pair[1] - index_pair[0])


    def forward(self, batch_data):
        # 对每个实体节点，聚合它邻居节点的信息，得到新的实体表示
        ent_batch_graph = EntBatchGraph(batch_data, self.meta_data).to(self.device_marker.device)
        x = ent_batch_graph.x['re']
        x = self.GAT_net(x, ent_batch_graph.edge_index) #(B, E, H)
        return x


In [6]:
import torch
import torch.nn as nn
from HolisticRCA.util.ent_batch_graph import EntBatchGraph


class FaultClassifier(nn.Module):
    def __init__(self, param_dict, meta_data):
        super().__init__()
        self.device_marker = nn.Parameter(torch.empty(0))
        self.meta_data = meta_data
        self.linear_dict = nn.ModuleDict()
        for ent_type in self.meta_data['ent_types']:
            index_pair = self.meta_data['ent_fault_type_index'][ent_type]
            self.linear_dict[ent_type] = nn.Linear(param_dict['eff_GAT_out_channels'], index_pair[1] - index_pair[0])

    def forward(self, x):
        output = dict()
        #(B, E, H)
        for ent_type in self.meta_data['ent_types']:
            # 切分不同实体类型
            temp = x[:, self.meta_data['ent_type_index'][ent_type][0]:self.meta_data['ent_type_index'][ent_type][1], :]
            # (B, n_ent, H) --> (B*n_ent, H)
            temp = temp.reshape(temp.shape[0] * temp.shape[1], temp.shape[2]).contiguous()
            # (B*n_ent, H) --> (B*n_ent, failure_type)
            output[ent_type] = self.linear_dict[ent_type](temp)
        return output


In [7]:
seed_everything()
torch.use_deterministic_algorithms(True)


# -------------------------
# Dataset wrapper
# -------------------------
class RCADataset(Dataset):
    def __init__(self, data_dict):
        self.data = data_dict
        self.n = len(self.data["y"])

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        item = {}
        for k, v in self.data.items():
            if k == "ent_edge_index":
                item[k] = torch.as_tensor(v[idx], dtype=torch.long)
            else:
                # x_*: float, y: float
                arr = np.asarray(v[idx])
                if k == "y":
                    item[k] = torch.as_tensor(arr, dtype=torch.long)
                else:
                    item[k] = torch.as_tensor(arr, dtype=torch.float32)
        return item


class RCADataLoader:
    def __init__(self, dataset_path, batch_size, shuffle_train=True):
        with open(dataset_path, "rb") as f:
            obj = pickle.load(f)

        self.meta = obj["meta_data"]
        self.data_loader = {}

        for split in ["train", "valid", "test"]:
            data = {}
            for m in self.meta["modal_types"]:
                x = obj["data"][f"x_{m}_{split}"] # (N, T, F)
                data[f"x_{m}"] = x.transpose((0, 2, 1))  # (N, F, T)

            data["ent_edge_index"] = obj["data"][f"ent_edge_index_{split}"] # List[N]
            data["y"] = obj["data"][f"y_{split}"]  # (N, E)

            ds = RCADataset(data)
            self.data_loader[split] = DataLoader(
                ds, batch_size=batch_size, shuffle=(split == "train" and shuffle_train)
            )


# -------------------------
# Loss helper: per-ent_type pos_weight
# -------------------------
def build_pos_weight_tensors(meta, device):
    # meta["ent_fault_type_weight"][ent_type] = list(K,)
    out = {}
    for ent_type in meta["ent_types"]:
        w = meta.get("ent_fault_type_weight", {}).get(ent_type, None)
        if w is None:
            out[ent_type] = None
        else:
            out[ent_type] = torch.tensor(w, dtype=torch.float32, device=device)
    return out


# -------------------------
# Stage-1 Trainer
# -------------------------
class Stage1Trainer:
    def __init__(self, params):
        self.params = params
        self.logger = Logger(logging_level="DEBUG").logger
        self.device = torch.device("cuda" if torch.cuda.is_available() and params["gpu"] else "cpu")

        self.dl = RCADataLoader(params["dataset_path"], params["batch_size"])
        self.meta = self.dl.meta

        # ---- build model (与你现在一致) ----
        o11y_representation_learning = RepresentationLearning(params, self.meta)
        re_feature_integration = FeatureIntegration(params, self.meta)
        re_feature_fusion = FeatureFusion(params, self.meta)
        re_fault_classifier = FaultClassifier(params, self.meta)
        # ent_type: (B*n_ent, failure_type)
        self.model = nn.Sequential(
            o11y_representation_learning,
            re_feature_integration,
            re_feature_fusion,
            re_fault_classifier
        ).to(self.device)
        print(self.model)

        self.pos_weight = build_pos_weight_tensors(self.meta, self.device)

    def _loss_one_batch(self, batch):
        # out: dict[ent_type] -> (B*n_ent, K)
        out = self.model(batch)

        # y_dict: dict[ent_type] -> (B*n_ent, K)
        y_dict = rearrange_y(self.meta, batch["y"], self.device)

        loss = 0.0
        used = 0
        for ent_type in self.meta["ent_types"]:
            y = y_dict[ent_type]
            logit = out[ent_type]

            # 过滤 mask=-1 的行
            keep = torch.where(~(y == -1).all(dim=1))[0]
            if keep.numel() == 0:
                continue

            # 对每个 ent_type 用自己的 pos_weight
            # crit = nn.BCEWithLogitsLoss()
            if self.pos_weight[ent_type] is None:
                crit = nn.BCEWithLogitsLoss()
            else:
                crit = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight[ent_type])

            loss = loss + crit(logit[keep], y[keep])
            used += 1

        if used == 0:
            return None
        return loss

    @torch.no_grad()
    def evaluate_split(self, split="valid"):
        self.model.eval()
        total_loss = 0.0
        n = 0
        for batch in self.dl.data_loader[split]:
            for k in batch:
                if k.startswith("x_") or k == "y":
                    batch[k] = batch[k].to(self.device)
            loss = self._loss_one_batch(batch)
            if loss is None:
                continue
            bs = batch["y"].shape[0]
            total_loss += float(loss.item()) * bs
            n += bs
        return total_loss / max(1, n)

    def train(self):
        opt = torch.optim.Adam(self.model.parameters(), lr=self.params["lr"], weight_decay=self.params["weight_decay"])

        best = 1e18
        for epoch in range(self.params["epochs"]):
            self.model.train()
            total = 0.0
            n = 0

            for batch in self.dl.data_loader["train"]:
        
                for k in batch:
                    if k.startswith("x_") or k == "y":
                        batch[k] = batch[k].to(self.device)
                '''
                    batch = {
                        "x_metric": tensor(B, F_metric, T),
                        "x_trace": tensor(B, F_trace, T),
                        "x_log": tensor(B, F_log, T),
                        "ent_edge_index": tensor(...),
                        "y": tensor(B, E)
                    }
                '''


                        
                opt.zero_grad()
                loss = self._loss_one_batch(batch)
                if loss is None:
                    continue
                loss.backward()
                opt.step()

                bs = batch["y"].shape[0]
                total += float(loss.item()) * bs
                n += bs

            train_loss = total / max(1, n)

            if epoch % 10 == 0:
                val_loss = self.evaluate_split("valid")
                self.logger.info(f"[{epoch}] train_loss={train_loss:.6f} valid_loss={val_loss:.6f}")


                y_pred, y_true = self.collect_pred_true(split="valid")
                self.output_evaluation_rca_d3_result(y_pred, y_true, "valid")

                # 你仍然可以保留一个 valid loss 作为 checkpoint 依据
                val_loss = self.evaluate_split("valid")
                
                if val_loss < best:
                    best = val_loss
                    torch.save(self.model.state_dict(), self.params["model_path"])
                    self.logger.info(f"[{epoch}] ✅ saved best -> {self.params['model_path']}")
        
        # 最终保存一次
        torch.save(self.model.state_dict(), self.params["model_path"])
        self.logger.info(f"done. saved -> {self.params['model_path']}")
        self.evaluate_rca_d3()

    @torch.no_grad()
    def collect_pred_true(self, split="valid"):
        self.model.eval()

        y_pred = {}
        y_true = {}

        for batch in self.dl.data_loader[split]:
            for k in batch:
                if k.startswith("x_") or k == "y":
                    batch[k] = batch[k].to(self.device)

            y = rearrange_y(self.meta, batch["y"], self.device)
            out = self.model(batch)

            for ent_type in self.meta["ent_types"]:
                fault_prob = torch.sigmoid(out[ent_type])

                # 源代码风格：按实体类型自己的阈值做二值化
                temp_y_pred = (
                    fault_prob > self.params[f"{ent_type}_accuracy_th"]
                ).cpu().detach().numpy()

                temp_y_true = y[ent_type].cpu().detach().numpy()

                if ent_type not in y_pred:
                    y_pred[ent_type] = []
                    y_true[ent_type] = []

                # ===== 与源代码完全一致的两种写法 =====
                if split == "valid":
                    # valid: 过滤 -1
                    valid_mask = temp_y_true != -1
                    y_pred[ent_type].extend(
                        temp_y_pred[valid_mask].reshape(-1, temp_y_pred.shape[1])
                    )
                    y_true[ent_type].extend(
                        temp_y_true[valid_mask].reshape(-1, temp_y_true.shape[1])
                    )
                else:
                    # test: 原源代码是直接 append 全量
                    y_pred[ent_type].extend(temp_y_pred)
                    y_true[ent_type].extend(temp_y_true)

        return y_pred, y_true

    def output_evaluation_rca_d3_result(self, y_pred, y_true, dataset_type):
        self.logger.info("----------")
        self.logger.info(f"evaluation dataset type: {dataset_type}")

        from sklearn.metrics import precision_recall_fscore_support
        
        convert = {
            "p": "precision",
            "r": "recall",
            "f1": "f1"
        }
        
        for ent_type in self.meta["ent_types"]:
            ent_y_pred = np.array(y_pred[ent_type])
            ent_y_true = np.array(y_true[ent_type])
        
            # weighted metrics
            p, r, f1, _ = precision_recall_fscore_support(
                ent_y_true,
                ent_y_pred,
                average="weighted",
                zero_division=0
            )
        
            self.logger.info(
                f'{ent_type.ljust(8)} | '
                f'weighted_precision: {p:.6f}; '
                f'weighted_recall: {r:.6f}; '
                f'weighted_f1: {f1:.6f}'
            )

        self.logger.info("----------")

    def evaluate_rca_d3(self):
        self.model.eval()
        y_pred, y_true = self.collect_pred_true(split="test")
        self.output_evaluation_rca_d3_result(y_pred, y_true, "test")




def _infer_K_from_meta(meta_data):
    if "fault_type_num" in meta_data and meta_data["fault_type_num"]:
        return int(meta_data["fault_type_num"])
    if "fault_type_list" in meta_data and meta_data["fault_type_list"]:
        return len(meta_data["fault_type_list"])
    if "ent_fault_type_index" in meta_data:
        a, b = meta_data["ent_fault_type_index"]["node"]
        return int(b - a)
    raise ValueError("Cannot infer K from meta_data.")



def rearrange_y(meta_data, y, device):
    """
    输入:
      y: (B, E) 或 (B, E, 1)
         值域:
           0    -> normal
           1..15 -> 全局 fault type id
           -1   -> 可选，表示无效/忽略

    输出:
      y_dict[ent_type]:
         node    -> (B*6, 6)
         service -> (B*10, 9)
         pod     -> (B*40, 9)

      这里是“局部 one-hot / all-zero”标签:
         normal -> 全零
         abnormal -> 对应局部 fault type 位置置 1
         invalid -> 全 -1
    """
    import torch

    if not torch.is_tensor(y):
        y = torch.as_tensor(y)

    if y.dim() == 3 and y.shape[-1] == 1:
        y = y.squeeze(-1)
    elif y.dim() != 2:
        raise ValueError(f"expect y shape (B,E) or (B,E,1), got {tuple(y.shape)}")

    y = y.to(device).long()
    B, E = y.shape

    ent_type_index = meta_data["ent_type_index"]
    ent_fault_type_index = meta_data["ent_fault_type_index"]

    y_dict = {}

    for ent_type in meta_data["ent_types"]:
        es, ee = ent_type_index[ent_type]          # 实体范围
        fs, fe = ent_fault_type_index[ent_type]    # fault type 局部范围（在 fault_type_list 中的 0-based slice）
        K_local = fe - fs

        # 当前实体类型对应的标签块: (B, n_ent)
        block = y[:, es:ee]

        # 输出局部 one-hot: (B, n_ent, K_local)
        out = torch.zeros((B, ee - es, K_local), dtype=torch.float32, device=device)

        # invalid: 全 -1
        invalid_mask = (block < 0)
        if invalid_mask.any():
            out[invalid_mask] = -1.0

        # valid abnormal: 全局类标 1..15
        # 对应到 fault_type_list 的 0-based 索引是 gid-1
        abnormal_mask = (block > 0)
        if abnormal_mask.any():
            global_zero_based = block[abnormal_mask] - 1   # 0..14

            # 检查这个实体类型的标签是否落在允许范围内
            local_mask = (global_zero_based >= fs) & (global_zero_based < fe)
            if not local_mask.all():
                bad_vals = torch.unique(block[abnormal_mask][~local_mask]).detach().cpu().tolist()
                raise ValueError(
                    f"found invalid fault ids for ent_type={ent_type}: {bad_vals}, "
                    f"allowed global zero-based range=[{fs},{fe})"
                )

            local_idx = global_zero_based - fs  # 映射到 0..K_local-1

            b_idx, e_local = torch.where(abnormal_mask)
            out[b_idx, e_local, local_idx] = 1.0

        # reshape 成 (B*n_ent, K_local)
        y_dict[ent_type] = out.reshape(B * (ee - es), K_local).contiguous()

    return y_dict




data_base_path = '../../../../HolisticRCA'
window_size=10 
dataset_path = f'{data_base_path}/temp_data/aiops22/dataset/merge_multimodal/rca_multimodal_window_size_{window_size}.pkl'
model_base_path = FileHandler.set_folder(f'{data_base_path}/model/aiops22')
checkpoint_dir = FileHandler.set_folder(model_base_path + "/checkpoint")

aiops22_params = { 
    "dataset_path": dataset_path, 
    "model_path": f"{checkpoint_dir}/main.pt", 
    "window_size": window_size, 
    "gpu": True, "epochs": 300, 
    "batch_size": 64, 
    "lr": 1e-4, 
    "weight_decay": 1e-3, 
    "node_accuracy_th": 0.5, 
    "service_accuracy_th": 0.5, 
    "pod_accuracy_th": 0.5, 
    "orl_te_heads": 2, 
    "orl_te_layers": 2, 
    "orl_te_in_channels": 256, 
    "efi_in_dim": 256, 
    "efi_te_heads": 4, 
    "efi_te_layers": 2, 
    "efi_out_dim": 256, 
    "eff_in_dim": 256, 
    "eff_GAT_out_channels": 128, 
    "eff_GAT_heads": 2, 
    "eff_GAT_dropout": 0.1, 
    "ec_fault_types": 15, 
    "explainer_gpu": True, 
    "explainer_epochs": 100, 
    "explainer_lr": 0.1, 
    "explainer_weight_decay": 0.001, 
    # ====== 关键：门控阈值（你论文的“先分类”门控）====== 
    "abnormal_gate_th": 0.5, 
    }




if __name__ == "__main__":
    # 直接复用你当前 aiops22_params
    trainer = Stage1Trainer(aiops22_params)
    trainer.train()

Sequential(
  (0): RepresentationLearning(
    (different_modal_mapping_dict): ModuleDict(
      (metric): Linear(in_features=10, out_features=256, bias=True)
      (trace): Linear(in_features=10, out_features=256, bias=True)
      (log): Linear(in_features=10, out_features=256, bias=True)
    )
    (positional_embedding_dict): ModuleDict(
      (metric): PositionalEmbedding()
      (trace): PositionalEmbedding()
      (log): PositionalEmbedding()
    )
    (modal_transformer_encoder_layer_dict): ModuleDict(
      (metric): TransformerEncoder(
        (layers): ModuleList(
          (0-1): 2 x TransformerEncoderLayer(
            (self_attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
            )
            (linear1): Linear(in_features=256, out_features=2048, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
            (linear2): Linear(in_features=2048, out_features=256, bias=True)
    

2026-03-08 21:52:19,374 - INFO - [0] train_loss=1.713837 valid_loss=1.211175
2026-03-08 21:52:19,473 - INFO - ----------
2026-03-08 21:52:19,474 - INFO - evaluation dataset type: valid
2026-03-08 21:52:19,477 - INFO - node     | weighted_precision: 0.000000; weighted_recall: 0.000000; weighted_f1: 0.000000
2026-03-08 21:52:19,478 - INFO - service  | weighted_precision: 0.000000; weighted_recall: 0.000000; weighted_f1: 0.000000
2026-03-08 21:52:19,481 - INFO - pod      | weighted_precision: 0.031825; weighted_recall: 0.284211; weighted_f1: 0.054368
2026-03-08 21:52:19,482 - INFO - ----------
2026-03-08 21:52:19,694 - INFO - [0] ✅ saved best -> ../../../../HolisticRCA/model/aiops22/checkpoint/main.pt
2026-03-08 21:52:37,779 - INFO - [10] train_loss=0.269298 valid_loss=0.311651
2026-03-08 21:52:37,877 - INFO - ----------
2026-03-08 21:52:37,879 - INFO - evaluation dataset type: valid
2026-03-08 21:52:37,882 - INFO - node     | weighted_precision: 0.798160; weighted_recall: 0.863636; weigh

**Localizer**

In [2]:
import os
import math
import copy
import pickle
import numpy as np
from typing import Any, Dict, List, Tuple, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn import Parameter, ParameterDict
from torch_geometric.nn import MessagePassing

from shared_util.seed import seed_everything
from shared_util.logger import Logger
from shared_util.file_handler import FileHandler

from HolisticRCA.model.common.GAT_net import GATNet
from HolisticRCA.util.ent_batch_graph import EntBatchGraph


In [9]:
def load_pkl(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)


def save_pkl(obj: Any, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=4)


def copy_batch_data(batch_data: Dict[str, Any], device: Optional[torch.device] = None):
    out = {}
    for k, v in batch_data.items():
        if torch.is_tensor(v):
            out[k] = v.clone()
            if device is not None:
                out[k] = out[k].to(device)
        else:
            out[k] = copy.deepcopy(v)
    return out


def safe_sigmoid_np(x: torch.Tensor) -> np.ndarray:
    return torch.sigmoid(x).detach().cpu().numpy()

def rearrange_y_for_localizer(meta_data, y, device):
    if not torch.is_tensor(y):
        y = torch.as_tensor(y)

    if y.dim() == 3 and y.shape[-1] == 1:
        y = y.squeeze(-1)
    elif y.dim() != 2:
        raise ValueError(f"expect y shape (B,E) or (B,E,1), got {tuple(y.shape)}")

    y = y.to(device).long()
    ent_type_index = meta_data["ent_type_index"]
    ent_fault_type_index = meta_data["ent_fault_type_index"]

    y_dict = {}

    for ent_type in meta_data["ent_types"]:
        es, ee = ent_type_index[ent_type]
        fs, fe = ent_fault_type_index[ent_type]
        K_local = fe - fs

        block = y[:, es:ee]  # (B, n_ent)
        out = torch.zeros((block.shape[0], block.shape[1], K_local), dtype=torch.float32, device=device)

        invalid_mask = (block < 0)
        if invalid_mask.any():
            out[invalid_mask] = -1.0

        abnormal_mask = (block > 0)
        if abnormal_mask.any():
            global_zero_based = block[abnormal_mask] - 1
            local_mask = (global_zero_based >= fs) & (global_zero_based < fe)
            if not local_mask.all():
                bad_vals = torch.unique(block[abnormal_mask][~local_mask]).detach().cpu().tolist()
                raise ValueError(
                    f"found invalid fault ids for ent_type={ent_type}: {bad_vals}, "
                    f"allowed global zero-based range=[{fs},{fe})"
                )

            local_idx = global_zero_based - fs
            b_idx, e_local = torch.where(abnormal_mask)
            out[b_idx, e_local, local_idx] = 1.0

        y_dict[ent_type] = out

    return y_dict

In [10]:
class PositionalEmbedding(nn.Module):
    def __init__(self, in_features, num_of_o11y_features, register_name):
        super().__init__()

        temp_in_features = in_features
        if temp_in_features % 2 == 1:
            temp_in_features += 1

        temp_num_of_o11y_features = num_of_o11y_features
        if temp_num_of_o11y_features % 2 == 1:
            temp_num_of_o11y_features += 1

        pe = torch.zeros(temp_in_features, temp_num_of_o11y_features).float()
        pe.require_grad = False

        position = torch.arange(0, temp_in_features).float().unsqueeze(1)
        div_term = (
            torch.arange(0, temp_num_of_o11y_features, 2).float()
            * -(math.log(10000.0) / temp_num_of_o11y_features)
        ).exp()

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.transpose(1, 0)[:num_of_o11y_features, :in_features]
        self.register_buffer("pe", pe)

    def forward(self, x):
        return self.pe + x


class RepresentationLearning(nn.Module):
    def __init__(self, param_dict, meta_data):
        super().__init__()
        self.device_marker = nn.Parameter(torch.empty(0))
        self.meta_data = meta_data
        self.different_modal_mapping_dict = nn.ModuleDict()
        self.positional_embedding_dict = nn.ModuleDict()
        self.modal_transformer_encoder_layer_dict = nn.ModuleDict()

        for modal_type in self.meta_data["modal_types"]:
            self.positional_embedding_dict[modal_type] = PositionalEmbedding(
                in_features=param_dict["window_size"],
                num_of_o11y_features=self.meta_data["o11y_length"][modal_type],
                register_name=f"{modal_type}_pe",
            )
            self.different_modal_mapping_dict[modal_type] = nn.Linear(
                in_features=param_dict["window_size"],
                out_features=param_dict["orl_te_in_channels"],
            )
            transformer_encoder_layer = nn.TransformerEncoderLayer(
                d_model=param_dict["orl_te_in_channels"],
                nhead=param_dict["orl_te_heads"],
            )
            self.modal_transformer_encoder_layer_dict[modal_type] = nn.TransformerEncoder(
                transformer_encoder_layer,
                num_layers=param_dict["orl_te_layers"],
            )

    def forward(self, batch_data):
        for modal_type in self.meta_data["modal_types"]:
            batch_data[f"x_{modal_type}"] = batch_data[f"x_{modal_type}"].to(self.device_marker.device)
            batch_data[f"x_{modal_type}"] = self.positional_embedding_dict[modal_type](
                batch_data[f"x_{modal_type}"]
            ).contiguous()
            batch_data[f"x_{modal_type}"] = self.different_modal_mapping_dict[modal_type](
                batch_data[f"x_{modal_type}"]
            )
            batch_data[f"x_{modal_type}"] = self.modal_transformer_encoder_layer_dict[modal_type](
                batch_data[f"x_{modal_type}"]
            )
        return batch_data


class FeatureIntegration(nn.Module):
    def __init__(self, param_dict, meta_data):
        super().__init__()
        self.device_marker = nn.Parameter(torch.empty(0))
        self.meta_data = meta_data

        self.ent_transformer_encoder_layer_dict = nn.ModuleDict()
        self.ent_feature_align_dict = nn.ModuleDict()

        in_dim = param_dict["efi_in_dim"]

        for ent_type in self.meta_data["ent_types"]:
            all_ent_feature_length = 0
            for modal_type in self.meta_data["modal_types"]:
                all_ent_feature_length += self.meta_data["max_ent_feature_num"][ent_type][modal_type]

            transformer_encoder_layer = nn.TransformerEncoderLayer(
                d_model=in_dim,
                nhead=param_dict["efi_te_heads"],
            )
            self.ent_transformer_encoder_layer_dict[ent_type] = nn.TransformerEncoder(
                transformer_encoder_layer,
                num_layers=param_dict["efi_te_layers"],
            )

            self.ent_feature_align_dict[ent_type] = nn.Linear(
                all_ent_feature_length * in_dim,
                param_dict["efi_out_dim"],
            )

    def forward(self, batch_data):
        batch_size = batch_data["y"].shape[0]

        x_ent = []
        for ent_type in self.meta_data["ent_types"]:
            for ent_index in range(
                self.meta_data["ent_type_index"][ent_type][0],
                self.meta_data["ent_type_index"][ent_type][1],
            ):
                x = []
                for modal_type in self.meta_data["modal_types"]:
                    feature_index_pair = self.meta_data["ent_features"][modal_type][ent_index][1]
                    modal_data = batch_data[f"x_{modal_type}"][
                        :, feature_index_pair[0]:feature_index_pair[1], :
                    ]
                    padding = torch.zeros(
                        batch_size,
                        self.meta_data["max_ent_feature_num"][ent_type][modal_type] - modal_data.shape[1],
                        modal_data.shape[2],
                    ).to(self.device_marker.device)
                    modal_data = torch.cat((modal_data, padding), 1)
                    x.append(modal_data)

                x = torch.cat(x, dim=1)
                x = self.ent_transformer_encoder_layer_dict[ent_type](x)
                x = x.view(batch_size, x.shape[1] * x.shape[2]).contiguous()
                x = self.ent_feature_align_dict[ent_type](x)
                x_ent.append(x)

        x_ent = torch.stack(x_ent, dim=1)
        batch_data["x_ent"] = x_ent
        return batch_data


class FeatureFusion(nn.Module):
    def __init__(self, param_dict, meta_data):
        super().__init__()
        self.device_marker = nn.Parameter(torch.empty(0))
        self.meta_data = meta_data
        self.GAT_net = GATNet(
            in_channels=param_dict["eff_in_dim"],
            out_channels=param_dict["eff_GAT_out_channels"],
            heads=param_dict["eff_GAT_heads"],
            dropout=param_dict["eff_GAT_dropout"],
        )
        self.linear_dict = nn.ModuleDict()
        for ent_type in self.meta_data["ent_types"]:
            index_pair = self.meta_data["ent_fault_type_index"][ent_type]
            self.linear_dict[ent_type] = nn.Linear(
                param_dict["eff_GAT_out_channels"],
                index_pair[1] - index_pair[0]
            )

    def forward(self, batch_data):
        ent_batch_graph = EntBatchGraph(batch_data, self.meta_data).to(self.device_marker.device)
        x = ent_batch_graph.x["re"]
        x = self.GAT_net(x, ent_batch_graph.edge_index)
        return x


class FaultClassifier(nn.Module):
    def __init__(self, param_dict, meta_data):
        super().__init__()
        self.device_marker = nn.Parameter(torch.empty(0))
        self.meta_data = meta_data
        self.linear_dict = nn.ModuleDict()
        for ent_type in self.meta_data["ent_types"]:
            index_pair = self.meta_data["ent_fault_type_index"][ent_type]
            self.linear_dict[ent_type] = nn.Linear(
                param_dict["eff_GAT_out_channels"],
                index_pair[1] - index_pair[0],
            )

    def forward(self, x):
        output = dict()
        for ent_type in self.meta_data["ent_types"]:
            temp = x[
                :,
                self.meta_data["ent_type_index"][ent_type][0]:self.meta_data["ent_type_index"][ent_type][1],
                :
            ]
            temp = temp.reshape(temp.shape[0] * temp.shape[1], temp.shape[2]).contiguous()
            output[ent_type] = self.linear_dict[ent_type](temp)
        return output


# class HolisticRCAModel(nn.Module):
#     """
#     不用 nn.Sequential，显式命名，方便 explainer 访问 self.feature_fusion.GAT_net
#     """
#     def __init__(self, param_dict, meta_data):
#         super().__init__()
#         self.representation_learning = RepresentationLearning(param_dict, meta_data)
#         self.feature_integration = FeatureIntegration(param_dict, meta_data)
#         self.feature_fusion = FeatureFusion(param_dict, meta_data)
#         self.fault_classifier = FaultClassifier(param_dict, meta_data)

#     def forward(self, batch_data):
#         batch_data = self.representation_learning(batch_data)
#         batch_data = self.feature_integration(batch_data)
#         x = self.feature_fusion(batch_data)
#         out = self.fault_classifier(x)
#         return out


In [11]:
class RCALocalizerDataset(Dataset):
    def __init__(self, obj: Dict[str, Any], split: str):
        self.meta = obj["meta_data"]
        self.split = split
        self.data = {}

        for m in self.meta["modal_types"]:
            x = obj["data"][f"x_{m}_{split}"]        # (N,T,F)
            self.data[f"x_{m}"] = x.transpose((0, 2, 1))  # -> (N,F,T)

        self.data["ent_edge_index"] = obj["data"][f"ent_edge_index_{split}"]
        self.data["y"] = obj["data"][f"y_{split}"]

        for k in ["entity_type", "template", "cmdb_id", "root_cause_type"]:
            kk = f"{k}_{split}"
            if kk in obj["data"]:
                self.data[k] = obj["data"][kk]

        self.n = len(self.data["y"])

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        item = {}
        for k, v in self.data.items():
            if k == "ent_edge_index":
                item[k] = torch.as_tensor(v[idx], dtype=torch.long)
            elif k.startswith("x_"):
                item[k] = torch.as_tensor(np.asarray(v[idx]), dtype=torch.float32)
            elif k == "y":
                item[k] = torch.as_tensor(np.asarray(v[idx]), dtype=torch.long)
            else:
                item[k] = v[idx]
        return item


def localizer_collate_fn(batch):
    """
    batch_size=1 最稳，但这里支持一般 batch。
    """
    out = {}
    keys = batch[0].keys()
    for k in keys:
        vals = [b[k] for b in batch]
        if torch.is_tensor(vals[0]):
            out[k] = torch.stack(vals, dim=0)
        else:
            out[k] = vals
    return out


class RCALocalizerDataLoader:
    def __init__(self, dataset_path: str, batch_size: int = 1):
        obj = load_pkl(dataset_path)
        self.meta_data = obj["meta_data"]
        self.data_loader = {}
        for split in ["train", "valid", "test"]:
            ds = RCALocalizerDataset(obj, split)
            self.data_loader[split] = DataLoader(
                ds,
                batch_size=batch_size,
                shuffle=False,
                collate_fn=localizer_collate_fn,
            )


In [23]:
class Explainer(nn.Module):
    def __init__(self, model, meta_data, param_dict):
        super().__init__()
        self.coeffs = {
            "ent_edge_size": 0.005,
            "ent_edge_reduction": "sum",
            "o11y_size": 1.0,
            "o11y_reduction": "mean",
            "ent_edge_entropy": 1.0,
            "o11y_entropy": 0.1,
            "EPS": 1e-15,
        }
        self.device = torch.device(
            "cuda" if torch.cuda.is_available() and param_dict["explainer_gpu"] else "cpu"
        )
        self.model = model
        self.meta_data = meta_data
        self.param_dict = param_dict
        self.o11y_mask = self.hard_o11y_mask = None
        self.ent_edge_mask = self.hard_ent_edge_mask = None
        self.criterion = torch.nn.BCEWithLogitsLoss()
        self.logger = Logger(logging_level="DEBUG").logger

    def init_o11y_mask(self):
        # feature mask
        mask = ParameterDict()
        for modal_type in self.meta_data["modal_types"]:
            '''
                特征长度，设置均值为1，表示对特征的掩码(1不影响特征、0掩掉特征)
                meta_data["o11y_length"] =
                {
                  metric : F_metric
                  trace  : F_trace
                  log    : F_log
                }
            '''
            mask[modal_type] = Parameter(
                torch.FloatTensor(self.meta_data["o11y_length"][modal_type]).to(self.device)
            )
            std = 0.1
            with torch.no_grad():
                mask[modal_type].normal_(1.0, std)
        self.o11y_mask = mask

    def init_ent_edge_mask(self, test_sample_data):
        # 创建实体资源图中边的mask,(num_edges,)
        num_edges = test_sample_data["ent_edge_index"].shape[2]
        num_entities = len(self.meta_data["ent_names"])
        std = nn.init.calculate_gain("relu") * math.sqrt(2.0 / (num_entities + num_entities))
        mask = Parameter(torch.randn(num_edges, device=self.device) * std)
        self.ent_edge_mask = mask

    def set_masks(self, test_sample_data):
        edge_index = torch.squeeze(test_sample_data["ent_edge_index"], dim=0)
        for module in self.model[2].GAT_net.modules():
            # 布尔mask，用于控制参与解释的边, (num_edges)
            loop_mask = torch.full_like(edge_index[0], True, dtype=torch.bool)
            # PyG MessagePassing 支持的 explainer机制
            # _loop_mask控制每条边的消息权重
            # _loop_mask控制哪些边允许应用 mask / 参与解释
            if isinstance(module, MessagePassing):
                module.explain = True
                module._edge_mask = self.ent_edge_mask
                module._loop_mask = loop_mask
                module._apply_sigmoid = True

    def clean_explainer(self):
        for module in self.model[2].GAT_net.modules():
            if isinstance(module, MessagePassing):
                module.explain = False
                module._edge_mask = None
                module._loop_mask = None
                module._apply_sigmoid = True
        self.o11y_mask = self.hard_o11y_mask = None
        self.ent_edge_mask = self.hard_ent_edge_mask = None

    def apply_o11y_mask(self, batch_data):
        """
        batch_data[x_modal]: (B, F_modal, T)
        用 feature-wise mask 去乘第 2 维 F
        """
        for modal_type in self.meta_data["modal_types"]:
            m = self.o11y_mask[modal_type].sigmoid().view(1, -1, 1)
            batch_data[f"x_{modal_type}"] = batch_data[f"x_{modal_type}"] * m
        return batch_data

    def _loss(self, masked_pred, original_y_pred):
        """
        保持原来的 fault type prediction
        """
        # 让加了 mask 之后的预测，尽量接近原来的预测
        loss = self.criterion(masked_pred, original_y_pred.detach())

        # 边稀疏损失:防止解决全1导致掩码失效
        ent_edge_mask = self.ent_edge_mask.sigmoid()
        if self.coeffs["ent_edge_reduction"] == "sum":
            edge_reduce = ent_edge_mask.sum()
        else:
            edge_reduce = ent_edge_mask.mean()
        loss = loss + self.coeffs["ent_edge_size"] * edge_reduce

        # 边离散化/清晰化约束:让 edge mask 不要模棱两可，尽量接近 0 或 1
        ent_edge_entropy = -ent_edge_mask * torch.log(ent_edge_mask + self.coeffs["EPS"]) - \
                           (1 - ent_edge_mask) * torch.log(1 - ent_edge_mask + self.coeffs["EPS"])
        loss = loss + self.coeffs["ent_edge_entropy"] * ent_edge_entropy.mean()

        # 特征稀疏损
        o11y_reduce_list = []
        o11y_entropy_list = []
        for modal_type in self.meta_data["modal_types"]:
            m = self.o11y_mask[modal_type].sigmoid()
            if self.coeffs["o11y_reduction"] == "mean":
                o11y_reduce_list.append(m.mean())
            else:
                o11y_reduce_list.append(m.sum())

            # 特征熵约束/清晰化
            ent = -m * torch.log(m + self.coeffs["EPS"]) - (1 - m) * torch.log(1 - m + self.coeffs["EPS"])
            o11y_entropy_list.append(ent.mean())

        # 把特征损失和边损失联合起来
        loss = loss + self.coeffs["o11y_size"] * torch.stack(o11y_reduce_list).mean()
        loss = loss + self.coeffs["o11y_entropy"] * torch.stack(o11y_entropy_list).mean()

        return loss

    def train_explainer(self, test_sample_data, original_y_pred, entity_type, entity_index):
        """
        输入:
          original_y_pred: (K_local,) 这是 classifier 对某个 entity 的 logit
        输出:
          ent_name_result: [(score, ent_name), ...]
          o11y_name_result: [(score, o11y_name), ...]
        1️⃣ 用 edge mask 推断实体级根因（D1）
        2️⃣ 用 feature mask 推断观测特征级根因证据（D2）
        """
        # 特征掩码
        self.init_o11y_mask()
        # 消息传递掩码
        self.init_ent_edge_mask(test_sample_data)
        # 图神经网络掩码设置
        self.set_masks(test_sample_data)

        parameters = [self.ent_edge_mask]
        for key in self.o11y_mask.keys():
            parameters.append(self.o11y_mask[key])

        optimizer = torch.optim.Adam(
            parameters,
            lr=self.param_dict["explainer_lr"],
            weight_decay=self.param_dict["explainer_weight_decay"],
        )

        self.model.eval()

        for _ in range(self.param_dict["explainer_epochs"]):
            optimizer.zero_grad()

            # 给原始特征设置特征掩码
            masked_batch = copy_batch_data(test_sample_data, self.device)
            masked_batch = self.apply_o11y_mask(masked_batch)

            out = self.model(masked_batch)
            # out[entity_type] shape = (B*n_ent, K_local)
            pred = out[entity_type][entity_index]
            loss = self._loss(pred, original_y_pred)

            loss.backward()
            optimizer.step()

        # -------- produce explanation result --------
        # 可视化结果 (anomaly_score, feature_name)
        o11y_name_result = []
        for modal_type in self.meta_data["modal_types"]:
            m = self.o11y_mask[modal_type].sigmoid().detach().cpu().numpy()
            names = self.meta_data["o11y_names"][modal_type]
            for i, score in enumerate(m):
                o11y_name_result.append((float(score), names[i]))

        o11y_name_result = sorted(o11y_name_result, key=lambda x: x[0], reverse=True)

        # 可视化结果 (anomaly_score, ent_name)
        ent_scores = {name: 0.0 for name in self.meta_data["ent_names"]}
        edge_mask = self.ent_edge_mask.sigmoid().detach().cpu().numpy()
        edge_index = torch.squeeze(test_sample_data["ent_edge_index"], dim=0).detach().cpu().numpy()

        trigger_global_idx = self.meta_data["ent_type_index"][entity_type][0] + entity_index
        
        for ei in range(edge_index.shape[1]):
            src = int(edge_index[0, ei])
            dst = int(edge_index[1, ei])
            score = float(edge_mask[ei])
            # ent_scores[self.meta_data["ent_names"][src]] += score
            # ent_scores[self.meta_data["ent_names"][dst]] += score

            if src == trigger_global_idx:
                ent_scores[self.meta_data["ent_names"][dst]] += score
        
            if dst == trigger_global_idx:
                ent_scores[self.meta_data["ent_names"][src]] += score

        ent_name_result = sorted([(v, k) for k, v in ent_scores.items()], key=lambda x: x[0], reverse=True)

        self.clean_explainer()
        return ent_name_result, o11y_name_result




In [24]:
class Stage2Localizer:
    def __init__(self, params):
        self.params = params
        self.logger = Logger(logging_level="DEBUG").logger
        self.device = torch.device("cuda" if torch.cuda.is_available() and params["gpu"] else "cpu")

        self.rca_data_loader = RCALocalizerDataLoader(
            dataset_path=params["dataset_path"],
            batch_size=1,   # 解释器必须逐样本
        )
        self.meta_data = self.rca_data_loader.meta_data

        o11y_representation_learning = RepresentationLearning(params, self.meta_data)
        re_feature_integration = FeatureIntegration(params, self.meta_data)
        re_feature_fusion = FeatureFusion(params, self.meta_data)
        re_fault_classifier = FaultClassifier(params, self.meta_data)
        
        self.model = nn.Sequential(
            o11y_representation_learning,
            re_feature_integration,
            re_feature_fusion,
            re_fault_classifier
        ).to(self.device)
        # self.model = HolisticRCAModel(self.params, self.meta_data).to(self.device)

    def _build_root_cause_list(self, batch_data):
        y_dict = rearrange_y_for_localizer(self.meta_data, batch_data["y"], self.device)
        # dict[ent_type] → (B, n_ent, K_local) n_ent = 该类型实体数 K_local = 该类型的 fault type 数量
        '''
            node    → (B, 6, 6)
            service → (B, 10, 9)
            pod     → (B, 40, 9)
        '''
        root_cause_list = []

        for ent_type in y_dict.keys():
            # 找出发生错误的[batch_idx, ent_idx, fault_idx]
            pos = torch.nonzero(y_dict[ent_type] > 0, as_tuple=False)
            for item in pos:
                b, ent_local_idx, fault_local_idx = int(item[0]), int(item[1]), int(item[2])
                if b != 0:
                    continue

                # 转换为全局编号
                ent_global_idx = self.meta_data["ent_type_index"][ent_type][0] + ent_local_idx
                fault_global_idx = self.meta_data["ent_fault_type_index"][ent_type][0] + fault_local_idx

                fault_name = self.meta_data["fault_type_list"][fault_global_idx]

                related_o11y = self.meta_data["fault_type_related_o11y_names"].get(fault_name, [])
                if isinstance(related_o11y, dict):
                    d2_value = related_o11y
                else:
                    d2_value = {
                        "exact": list(related_o11y),
                        "fuzzy": list(related_o11y),
                    }


                
                root_cause_list.append({
                    "d1": self.meta_data["ent_names"][ent_global_idx],
                    "d2": d2_value,
                    "level": ent_type,
                    "fault_type": fault_name,
                })

        # 因为batch_size=1，之前标注的时候把service的同时给pod标注了，所以这里相当于还原真实的标注
        exact_root_cause = {}
        for root_cause in root_cause_list:
            exact_root_cause = root_cause
            if root_cause["level"] == "service":
                break

        return root_cause_list, exact_root_cause

    def _compute_hit(self, localization_result, exact_root_cause):
        d1_hit = {1: False, 3: False, 5: False}
        d2_hit = {1: False, 3: False, 5: False}

        for i in range(len(localization_result["d1"])):
            pred_name = localization_result["d1"][i][0]
            if exact_root_cause["d1"] in pred_name:
                for k in [1, 3, 5]:
                    if i < k:
                        d1_hit[k] = True

        exact_o11y = exact_root_cause["d2"]
        for i in range(len(localization_result["d2"])):
            pred_o11y = localization_result["d2"][i][0]
            hit = False

            for exact_name in exact_o11y["exact"]:
                if exact_root_cause["d1"] in pred_o11y and exact_name in pred_o11y:
                    hit = True

            for fuzzy_name in exact_o11y["fuzzy"]:
                if fuzzy_name in pred_o11y:
                    hit = True

            if hit:
                for k in [1, 3, 5]:
                    if i < k:
                        d2_hit[k] = True

        return d1_hit, d2_hit

    def predict(self):
        # 加载错误分类阶段训练好的模型
        self.model.load_state_dict(torch.load(self.params["model_path"], map_location=self.device))
        self.model.eval()

        explainer = Explainer(self.model, self.meta_data, self.params)

        result = {}
        for ent_type in self.meta_data["ent_types"]:
            result[ent_type] = {
                "total": 0,
                "d1": {"AC@1_num": 0, "AC@3_num": 0, "AC@5_num": 0},
                "d2": {"AC@1_num": 0, "AC@3_num": 0, "AC@5_num": 0},
            }

        detail_result = []

        for batch_id, batch_data in enumerate(self.rca_data_loader.data_loader["test"]):
            explain_batch_data = copy_batch_data(batch_data, self.device)

            # 模型预测也要 device
            infer_batch_data = copy_batch_data(batch_data, self.device)

            root_cause_list, exact_root_cause = self._build_root_cause_list(infer_batch_data)
            if len(root_cause_list) == 0:
                continue

            '''
            exact_root_cause:
                {
                 d1 : "checkoutservice",
                 d2 :
                 {
                   exact : ["checkoutservice/cpu_usage"],
                   fuzzy : ["cpu"]
                 },
                 level : "service",
                 fault_type : "container_cpu_load"
                }

            '''
            result[exact_root_cause["level"]]["total"] += 1

            out = self.model(infer_batch_data)

            localization_result = {
                "d1": {},
                "d2": {},
            }

            suspect_list = []

            for ent_type in self.meta_data["ent_types"]:
                # (B * n_ent_type, K), 每个值为布尔值0/1表示是否为错误
                temp_y_pred = (
                    torch.sigmoid(out[ent_type]) > self.params[f"{ent_type}_accuracy_th"]
                ).cpu().detach().numpy()

                # 计算实体异常概率（预测为几类异常的概率值中去最大的概率值）
                fault_prob = torch.sigmoid(out[ent_type])
                ent_fault_prob = torch.max(fault_prob, dim=1).values.cpu().detach().numpy()

                for ent_index in range(len(temp_y_pred)):
                    # (ent_index, ent_type, anomaly_score)
                    suspect_list.append((ent_index, ent_type, ent_fault_prob[ent_index]))

                    # 如果这个实体被判定为异常
                    if temp_y_pred[ent_index].any():
                        suspect_list.pop() # 移除

                        # 转为全局的实体编号，d1设置为触发异常实体的异常得分
                        trigger_ent = self.meta_data["ent_names"][
                            self.meta_data["ent_type_index"][ent_type][0] + ent_index
                        ]
                        localization_result["d1"].setdefault(trigger_ent, 0.0)
                        localization_result["d1"][trigger_ent] += float(ent_fault_prob[ent_index])

                        ent_name_result, o11y_name_result = explainer.train_explainer(
                            explain_batch_data,
                            fault_prob[ent_index],
                            ent_type,
                            ent_index,
                        )

                        for score, ent_name in ent_name_result:
                            localization_result["d1"].setdefault(ent_name, 0.0)
                            # 解释器给出的实体重要性×该实体触发异常的概率
                            localization_result["d1"][ent_name] += float(score) * float(ent_fault_prob[ent_index])

                        for score, o11y_name in o11y_name_result:
                            localization_result["d2"].setdefault(o11y_name, 0.0)
                            # 解释器给出的特征重要性×实体触发异常的概率
                            localization_result["d2"][o11y_name] += float(score) * float(ent_fault_prob[ent_index])


            # 1.先把所有实体当作 suspect 2. 如果该实体已经被预测为异常 3. 就把它从 suspect_list 删除
            # 如果解释结果还不够（D1 / D2 < 5）,如果解释结果还不够（D1 / D2 < 5）
            suspect_index = 0
            suspect_list = sorted(suspect_list, key=lambda item: item[2], reverse=True)

            while (len(localization_result["d1"]) < 5 or len(localization_result["d2"]) < 5) and suspect_index < len(suspect_list):
                ent_index, ent_type, ent_fault_prob = suspect_list[suspect_index]

                ent_name_result, o11y_name_result = explainer.train_explainer(
                    explain_batch_data,
                    torch.sigmoid(out[ent_type])[ent_index],
                    ent_type,
                    ent_index,
                )

                for score, ent_name in ent_name_result:
                    localization_result["d1"].setdefault(ent_name, 0.0)
                    localization_result["d1"][ent_name] += float(score) * float(ent_fault_prob)

                for score, o11y_name in o11y_name_result:
                    localization_result["d2"].setdefault(o11y_name, 0.0)
                    localization_result["d2"][o11y_name] += float(score) * float(ent_fault_prob)

                suspect_index += 1

            localization_result["d1"] = sorted(
                localization_result["d1"].items(),
                key=lambda item: item[1],
                reverse=True,
            )
            localization_result["d2"] = sorted(
                localization_result["d2"].items(),
                key=lambda item: item[1],
                reverse=True,
            )

            d1_hit, d2_hit = self._compute_hit(localization_result, exact_root_cause)

            for k in [1, 3, 5]:
                if d1_hit[k]:
                    result[exact_root_cause["level"]]["d1"][f"AC@{k}_num"] += 1
                if d2_hit[k]:
                    result[exact_root_cause["level"]]["d2"][f"AC@{k}_num"] += 1

            detail_result.append({
                "sample_id": batch_id,
                "exact_root_cause": exact_root_cause,
                "predict_d1": localization_result["d1"],
                "predict_d2": localization_result["d2"],
                "extra": {
                    "entity_type": batch_data["entity_type"][0] if "entity_type" in batch_data else None,
                    "template": batch_data["template"][0] if "template" in batch_data else None,
                    "cmdb_id": batch_data["cmdb_id"][0] if "cmdb_id" in batch_data else None,
                    "root_cause_type": batch_data["root_cause_type"][0] if "root_cause_type" in batch_data else None,
                }
            })

            self.logger.info("----------")
            self.logger.info(
                f'[{batch_id}] gt d1={exact_root_cause["d1"]}, '
                f'level={exact_root_cause["level"]}, '
                f'fault_type={exact_root_cause["fault_type"]}'
            )
            self.logger.info(f'[{batch_id}] pred d1 top5={localization_result["d1"][:5]}')
            self.logger.info(f'[{batch_id}] pred d2 top5={localization_result["d2"][:5]}')
            self.logger.info("----------")

        self.logger.info("========== FINAL RESULT ==========")
        for ent_type in result.keys():
            total = max(result[ent_type]["total"], 1)
            self.logger.info(
                f'{ent_type} | d1 AC@1={result[ent_type]["d1"]["AC@1_num"]/total:.6f} '
                f'AC@3={result[ent_type]["d1"]["AC@3_num"]/total:.6f} '
                f'AC@5={result[ent_type]["d1"]["AC@5_num"]/total:.6f}'
            )
            self.logger.info(
                f'{ent_type} | d2 AC@1={result[ent_type]["d2"]["AC@1_num"]/total:.6f} '
                f'AC@3={result[ent_type]["d2"]["AC@3_num"]/total:.6f} '
                f'AC@5={result[ent_type]["d2"]["AC@5_num"]/total:.6f}'
            )

        if "output_path" in self.params:
            save_pkl(
                {
                    "summary": result,
                    "detail": detail_result,
                },
                self.params["output_path"],
            )
            self.logger.info(f'[OK] saved -> {self.params["output_path"]}')

        return {
            "summary": result,
            "detail": detail_result,
        }

In [25]:
if __name__ == "__main__":
    seed_everything()
    torch.use_deterministic_algorithms(True)

    data_base_path = "../../../../HolisticRCA"
    window_size = 10

    dataset_path = f"{data_base_path}/temp_data/aiops22/dataset/merge_multimodal/rca_multimodal_window_size_{window_size}.pkl"
    model_base_path = FileHandler.set_folder(f"{data_base_path}/model/aiops22")
    checkpoint_dir = FileHandler.set_folder(model_base_path + "/checkpoint")
    localization_dir = FileHandler.set_folder(model_base_path + "/localization")

    aiops22_params = {
        "dataset_path": dataset_path,
        "model_path": f"{checkpoint_dir}/main.pt",
        "output_path": f"{localization_dir}/localization_window_size_{window_size}.pkl",

        "window_size": window_size,

        "gpu": True,
        "batch_size": 1,

        "node_accuracy_th": 0.5,
        "service_accuracy_th": 0.5,
        "pod_accuracy_th": 0.5,

        "orl_te_heads": 2,
        "orl_te_layers": 2,
        "orl_te_in_channels": 256,

        "efi_in_dim": 256,
        "efi_te_heads": 4,
        "efi_te_layers": 2,
        "efi_out_dim": 256,

        "eff_in_dim": 256,
        "eff_GAT_out_channels": 128,
        "eff_GAT_heads": 2,
        "eff_GAT_dropout": 0.1,

        "ec_fault_types": 15,

        "explainer_gpu": True,
        "explainer_epochs": 100,
        "explainer_lr": 0.1,
        "explainer_weight_decay": 0.001,
    }

    localizer = Stage2Localizer(aiops22_params)
    localizer.predict()

2026-03-09 14:38:47,394 - INFO - ----------
2026-03-09 14:38:47,396 - INFO - [0] gt d1=node-6, level=node, fault_type=node_cpu_failure
2026-03-09 14:38:47,396 - INFO - [0] pred d1 top5=[('node-2', 0.012095424935843369), ('node-1', 0.0), ('node-3', 0.0), ('node-4', 0.0), ('node-5', 0.0)]
2026-03-09 14:38:47,397 - INFO - [0] pred d2 top5=[('node-2/CPU/system.load.1', 0.023251382191659786), ('node-2/Disk/system.io.svctm', 0.022296194008635695), ('node-2/Memory/system.mem.pct_usage', 0.02222541540905354), ('node-2/CPU/system.cpu.system', 0.02218128830678978), ('node-2/Disk/system.disk.pct_usage', 0.02216822376666272)]
2026-03-09 14:38:47,397 - INFO - ----------
2026-03-09 14:39:08,449 - INFO - ----------
2026-03-09 14:39:08,450 - INFO - [1] gt d1=node-6, level=node, fault_type=node_cpu_failure
2026-03-09 14:39:08,451 - INFO - [1] pred d1 top5=[('node-2', 0.017891365119286462), ('node-1', 0.0), ('node-3', 0.0), ('node-4', 0.0), ('node-5', 0.0)]
2026-03-09 14:39:08,451 - INFO - [1] pred d2 t